# Build Word2Vec Tables

This notebook trains one word2vec embedding model on the full constitution corpus, exports a `VOCAB_W2V` table, and creates a reusable t-SNE coordinate table for plotting.

## Inputs and Outputs

**Input files**
- `corpus.csv`
- `vocab.csv`
- `lib.csv`

**Output files**
- `vocab_w2v.parquet`
- `vocab_w2v_tsne.parquet`

This notebook uses article-level bags as the word2vec training documents. The constitution corpus does not preserve a sentence id, and article-level bags usually give a better balance of context and stability for this corpus than very short clause-level bags.

That choice is justified because the corpus contains more than 33,000 article-level documents, so article bags provide enough context for stable embeddings without collapsing the whole constitution into one very broad usage window. The resulting embedding is still learned at the term level, though, so later metadata overlays should be treated as interpretive additions rather than as dimensions native to the embedding itself.


## Set Up

In [9]:
#  this notebook adapts the HW09 word2vec + t-SNE workflow from `uva-ds-5001-m09-hw.ipynb`.
# Keep the homework provenance here in code comments instead of the final-project-facing documentation.
from pathlib import Path

import pandas as pd
import plotly.express as px

try:
    from gensim.models import word2vec
except ModuleNotFoundError:
    word2vec = None

from sklearn.manifold import TSNE

CORPUS_CSV = Path('corpus.csv')
VOCAB_CSV = Path('vocab.csv')
LIB_CSV = Path('lib.csv')

VOCAB_W2V_PARQUET = Path('vocab_w2v.parquet')
VOCAB_W2V_TSNE_PARQUET = Path('vocab_w2v_tsne.parquet')
VOCAB_W2V_TSNE_HTML = Path('vocab_w2v_tsne.html')
VOCAB_W2V_TSNE_FOCUS_HTML = Path('vocab_w2v_tsne_focus.html')

OHCO = ['country_id', 'article_n', 'clause_n', 'token_n']
BAG_COLS = ['country_id', 'article_n']

# Start close to HW09, but use a lower min_count because constitutional vocabulary is more specialized.
W2V_PARAMS = dict(
    window=2,
    vector_size=256,
    min_count=20,
    workers=1,
    seed=42,
)
TSNE_PARAMS = dict(
    perplexity=20,
    n_components=2,
    init='pca',
    max_iter=1000,
    random_state=42,
)


## Load Inputs

The main corpus-specific choices here are:

- use `article_n` as the local document unit instead of `sent_num`
- build embeddings from normalized `term_str`
- respect the project `VOCAB.stop` filter so stopwords and legal boilerplate do not dominate the embedding space

In [10]:
if word2vec is None:
    raise ModuleNotFoundError('gensim is required for this notebook. Install it in the environment before running build_word2vec.ipynb.')

# Load only the fields needed for building article-level training documents.
CORPUS = pd.read_csv(CORPUS_CSV, usecols=OHCO[:-1] + ['token_n', 'term_str']).set_index(OHCO)
VOCAB = pd.read_csv(VOCAB_CSV).set_index('term_str').sort_index()
LIB = pd.read_csv(LIB_CSV).set_index('country_id').sort_index()

# Drop blank normalized terms and apply the project stopword/legal-boilerplate filter.
CORPUS['term_str'] = CORPUS['term_str'].fillna('').astype(str).str.strip()
CORPUS = CORPUS[CORPUS['term_str'] != ''].copy()
ACTIVE_TERMS = VOCAB[VOCAB['stop'] == 0].index
CORPUS = CORPUS[CORPUS['term_str'].isin(ACTIVE_TERMS)].copy()

print(f'Corpus token rows after filtering: {len(CORPUS):,}')
print(f'Active vocabulary size: {len(ACTIVE_TERMS):,}')
print(f'Countries represented: {CORPUS.index.get_level_values("country_id").nunique():,}')
CORPUS.head()

Corpus token rows after filtering: 2,108,387
Active vocabulary size: 24,681
Countries represented: 192


term_str
country_id  article_n clause_n token_n            
Afghanistan 1         1        3              name
                               5             allah
                               7              most
                               8        beneficent
                               10             most

## Build Article Documents

In [11]:
# Group normalized terms into article-level token lists for gensim.
ARTICLE_DOCS = CORPUS.groupby(BAG_COLS)['term_str'].apply(list)
DOCS = ARTICLE_DOCS.tolist()

print(f'Article-level documents: {len(DOCS):,}')
print(f'Min tokens per article: {ARTICLE_DOCS.str.len().min():,}')
print(f'Median tokens per article: {ARTICLE_DOCS.str.len().median():.0f}')
print('Sample article document:', DOCS[0][:30])

Article-level documents: 33,725
Min tokens per article: 1
Median tokens per article: 34
Sample article document: ['name', 'allah', 'most', 'beneficent', 'most', 'merciful', 'praise', 'allah', 'cherisher', 'sustainer', 'worlds', 'praise', 'peace', 'upon', 'mohammad', 'last', 'messenger', 'disciples', 'followers', 'we', 'people', 'afghanistan', 'believing', 'firmly', 'almighty', 'god', 'relying', 'divine', 'adhering', 'holy']


## Train Word2Vec

In [12]:
# Fit one constitution-wide embedding model using the HW09 parameter pattern.
W2V_MODEL = word2vec.Word2Vec(DOCS, **W2V_PARAMS)

# Export the learned vectors as a term-by-feature table aligned to term_str.
VOCAB_W2V = pd.DataFrame(W2V_MODEL.wv.vectors, index=W2V_MODEL.wv.index_to_key)
VOCAB_W2V.index.name = 'term_str'
VOCAB_W2V.columns = [f'w2v_{i}' for i in range(VOCAB_W2V.shape[1])]
VOCAB_W2V = VOCAB.join(VOCAB_W2V, how='inner')

print(f'Embedded vocabulary size: {VOCAB_W2V.shape[0]:,}')
print(f'Embedding dimensions: {W2V_PARAMS["vector_size"]}')
VOCAB_W2V.iloc[:5, :12]

Embedded vocabulary size: 5,622
Embedding dimensions: 256


,n,p,i,df,idf,dfidf,n_chars,max_pos,max_pos_group,n_pos,cat_pos,n_pos_group
term_str,,,,,,,,,,,,
abandoned,35,0.000009,16.743293,27,3.785102,102.197757,9,VBN,VB,8,"{IN, JJ, NN, NNP, NNPS, VB, VBD, VBN}",4
abandonment,40,0.000010,16.550648,32,3.548063,113.538013,11,NN,NN,4,"{JJ, NN, NNP, VB}",3
abeyance,35,0.000009,16.743293,5,6.007495,30.037473,8,NN,NN,3,"{CC, JJ, NN}",3
abide,104,0.000027,15.172136,51,2.892017,147.492883,5,VB,VB,4,"{NN, RB, VB, VBP}",3
abilities,52,0.000014,16.172136,41,3.200140,131.205724,9,NNS,NN,4,"{JJ, NN, NNS, VB}",3


## Build t-SNE Coordinates

This step projects the high-dimensional word2vec vectors into two dimensions for plotting. The reduction is useful because it turns the term-level embedding into a readable neighborhood map, but t-SNE is designed mainly to preserve local structure. Nearby words usually remain meaningful neighbors, while large visual distances between clusters should be interpreted more cautiously because global geometry can be distorted.


In [13]:
# Project the embedding table into two dimensions for exploratory plotting.
tsne_engine = TSNE(**TSNE_PARAMS)
VOCAB_W2V_TSNE = pd.DataFrame(
    tsne_engine.fit_transform(VOCAB_W2V.filter(like='w2v_')),
    columns=['x', 'y'],
    index=VOCAB_W2V.index,
)
VOCAB_W2V_TSNE.index.name = 'term_str'
VOCAB_W2V_TSNE = VOCAB_W2V[['n', 'df', 'i']].join(VOCAB_W2V_TSNE)

print(f't-SNE rows: {VOCAB_W2V_TSNE.shape[0]:,}')
VOCAB_W2V_TSNE.head()

t-SNE rows: 5,622


,n,df,i,x,y
term_str,,,,,
abandoned,35,27,16.743293,-17.639801,-20.022953
abandonment,40,32,16.550648,-22.354099,-11.046404
abeyance,35,5,16.743293,39.372463,15.915979
abide,104,51,15.172136,17.605333,-54.209518
abilities,52,41,16.172136,-34.124271,-10.093642


## Plot t-SNE

The t-SNE figure is best read as a semantic neighborhood display rather than an axis-based model summary. The plotted points are terms, not constitutions or articles, so any coloring by metadata has to be joined back in afterward through corpus usage rather than being learned directly by the embedding.


In [14]:
# Plot the full embedding space and highlight the rights/freedom neighborhood.
TSNE_PLOT = VOCAB_W2V_TSNE.reset_index().copy()
TSNE_PLOT['label_term'] = ''
TOP_LABEL_TERMS = TSNE_PLOT.nlargest(175, 'n')['term_str']
TSNE_PLOT.loc[TSNE_PLOT['term_str'].isin(TOP_LABEL_TERMS), 'label_term'] = TSNE_PLOT['term_str']

focus_terms = ['rights', 'freedom']
focus_points = TSNE_PLOT.loc[TSNE_PLOT['term_str'].isin(focus_terms)].copy()
missing_focus = sorted(set(focus_terms) - set(focus_points['term_str']))
W2V_TSNE_FIG = None
W2V_TSNE_FOCUS_FIG = None

if missing_focus:
    print(f'Skipping focus-box plot because these terms are missing from the vocabulary: {missing_focus}')
    fig = px.scatter(
        TSNE_PLOT,
        x='x',
        y='y',
        text='label_term',
        hover_name='term_str',
        hover_data={'n': True, 'df': True, 'i': ':.2f'},
        size='n',
        size_max=16,
        height=950,
        width=1200,
        title='Constitution Word2Vec t-SNE Space'
    )
    fig.update_traces(textposition='top center')
    fig.update_layout(template='plotly_white')
    W2V_TSNE_FIG = fig
    fig.show()
else:
    span_x = max(focus_points['x'].max() - focus_points['x'].min(), 1e-6)
    span_y = max(focus_points['y'].max() - focus_points['y'].min(), 1e-6)
    pad_x = max(span_x * 0.6, 4)
    pad_y = max(span_y * 0.6, 4)
    x0 = focus_points['x'].min() - pad_x
    x1 = focus_points['x'].max() + pad_x
    y0 = focus_points['y'].min() - pad_y
    y1 = focus_points['y'].max() + pad_y

    in_focus_box = TSNE_PLOT['x'].between(x0, x1) & TSNE_PLOT['y'].between(y0, y1)
    TSNE_PLOT['focus_group'] = 'Other terms'
    TSNE_PLOT.loc[in_focus_box, 'focus_group'] = 'Inside focus box'

    fig = px.scatter(
        TSNE_PLOT,
        x='x',
        y='y',
        color='focus_group',
        color_discrete_map={'Other terms': '#b8b8b8', 'Inside focus box': '#c23b22'},
        text='label_term',
        hover_name='term_str',
        hover_data={'n': True, 'df': True, 'i': ':.2f', 'focus_group': False},
        size='n',
        size_max=16,
        height=950,
        width=1200,
        title='Constitution Word2Vec t-SNE Space with expanded rights/freedom focus box'
    )
    fig.add_shape(
        type='rect',
        x0=x0,
        x1=x1,
        y0=y0,
        y1=y1,
        line=dict(color='#c23b22', width=3),
        fillcolor='rgba(194, 59, 34, 0.08)'
    )
    fig.update_traces(textposition='top center')
    fig.update_layout(template='plotly_white')
    W2V_TSNE_FIG = fig
    fig.show()

    focus_only = TSNE_PLOT.loc[in_focus_box].copy()
    focus_only['focus_label'] = focus_only['term_str']

    fig_focus_only = px.scatter(
        focus_only,
        x='x',
        y='y',
        text='focus_label',
        hover_name='term_str',
        hover_data={'n': True, 'df': True, 'i': ':.2f'},
        size='n',
        size_max=18,
        height=950,
        width=1200,
        title='Terms inside the rights/freedom focus box'
    )
    fig_focus_only.update_traces(marker=dict(color='#c23b22'), textposition='top center')
    fig_focus_only.update_layout(template='plotly_white')
    W2V_TSNE_FOCUS_FIG = fig_focus_only
    fig_focus_only.show()


## Quick Similarity Helpers

In [15]:
def get_most_similar(model, positive, negative=None, topn=10):
    # Mirror the HW09 exploration pattern for quick semantic checks.
    return pd.DataFrame(model.wv.most_similar(positive=positive, negative=negative, topn=topn), columns=['term', 'sim'])


def complete_analogy(model, A, B, C, topn=5):
    # Solve analogies in the usual word2vec vector form: (B - A) + C.
    return pd.DataFrame(model.wv.most_similar(positive=[B, C], negative=[A], topn=topn), columns=['term', 'sim'])

## Save Outputs

In [16]:
# Save the embedding tables and interactive plots for reuse in the final project notebook.
VOCAB_W2V.reset_index().to_parquet(VOCAB_W2V_PARQUET, index=False)
VOCAB_W2V_TSNE.reset_index().to_parquet(VOCAB_W2V_TSNE_PARQUET, index=False)
if W2V_TSNE_FIG is not None:
    W2V_TSNE_FIG.write_html(VOCAB_W2V_TSNE_HTML, include_plotlyjs='cdn')
if W2V_TSNE_FOCUS_FIG is not None:
    W2V_TSNE_FOCUS_FIG.write_html(VOCAB_W2V_TSNE_FOCUS_HTML, include_plotlyjs='cdn')

print(f'Saved VOCAB_W2V to: {VOCAB_W2V_PARQUET.resolve()}')
print(f'Saved VOCAB_W2V_TSNE to: {VOCAB_W2V_TSNE_PARQUET.resolve()}')
if W2V_TSNE_FIG is not None:
    print(f'Saved Word2Vec t-SNE HTML to: {VOCAB_W2V_TSNE_HTML.resolve()}')
if W2V_TSNE_FOCUS_FIG is not None:
    print(f'Saved focused Word2Vec t-SNE HTML to: {VOCAB_W2V_TSNE_FOCUS_HTML.resolve()}')

Saved VOCAB_W2V to: C:\Users\garre\school\spring_2026\ds_5001\vocab_w2v.parquet
Saved VOCAB_W2V_TSNE to: C:\Users\garre\school\spring_2026\ds_5001\vocab_w2v_tsne.parquet
Saved Word2Vec t-SNE HTML to: C:\Users\garre\school\spring_2026\ds_5001\vocab_w2v_tsne.html
Saved focused Word2Vec t-SNE HTML to: C:\Users\garre\school\spring_2026\ds_5001\vocab_w2v_tsne_focus.html
